1.Imports

In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.5.1
True


In [2]:
import sys
print(sys.executable)


d:\Anaconda\envs\resnet\python.exe


In [3]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

print("PyTorch version:", torch.__version__)

PyTorch version: 2.5.1


## Device

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


### 3. Dataset Path

In [5]:
dataset_path = "D:/Judy Uni/Semester 6/Deep Learning/deep project/NWPU-RESISC45"

### 4. Transforms

In [6]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### 6. load + split

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
import numpy as np


train_full = datasets.ImageFolder(
    root=dataset_path,
    transform=train_transforms
)

eval_full = datasets.ImageFolder(
    root=dataset_path,
    transform=val_test_transforms
)

print("Total images:", len(train_full))
print("Classes:", len(train_full.classes))

# CREATE SPLITS

dataset_size = len(train_full)

indices = list(range(dataset_size))
np.random.shuffle(indices)

train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

# CREATE SUBSETS

train_dataset = Subset(train_full, train_indices)

val_dataset = Subset(eval_full, val_indices)

test_dataset = Subset(eval_full, test_indices)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

Total images: 31500
Classes: 45
Train size: 22050
Validation size: 4725
Test size: 4725


### 7. Set Validation/Test Transforms

In [8]:
val_dataset.dataset.transform = val_test_transforms
test_dataset.dataset.transform = val_test_transforms

### 8. Create Dataloaders

In [9]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("DataLoaders ready!")

DataLoaders ready!


### 9. Load Pretrained ResNet18

In [10]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### 10. Replace Final Layer

In [11]:
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 45)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=45, bias=True)


### 11. Freeze Backbone

In [12]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

print("Backbone frozen!")
print("Only classifier will train.")

Backbone frozen!
Only classifier will train.


### 12. Loss Function and Optimizer

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

In [14]:
criterion = FocalLoss(gamma=2.0)

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

### 13. Training Function

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs,
    patience=5
):

    best_model_weights = copy.deepcopy(model.state_dict())
    best_accuracy = 0.0

    # Early stopping counter
    counter = 0

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 30)

        # TRAINING

        model.train()

        running_loss = 0.0
        running_corrects = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * images.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)

        print(f"Train Loss: {epoch_loss:.4f}")
        print(f"Train Acc : {epoch_acc:.4f}")

        # VALIDATION

        model.eval()

        val_loss = 0.0
        val_corrects = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, preds = torch.max(outputs, 1)

                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                val_corrects += torch.sum(preds == labels.data)

        val_epoch_loss = val_loss / len(val_loader.dataset)
        val_epoch_acc = val_corrects.double() / len(val_loader.dataset)

        print(f"Val Loss : {val_epoch_loss:.4f}")
        print(f"Val Acc  : {val_epoch_acc:.4f}")

        # SAVE BEST MODEL

        if val_epoch_acc > best_accuracy:

            best_accuracy = val_epoch_acc

            best_model_weights = copy.deepcopy(model.state_dict())

            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "val_acc": float(val_epoch_acc)
                },
                "best_pretrained_resnet18.pth"
            )

            print("Best model saved!")

            # reset early stopping counter
            counter = 0

        else:

            counter += 1

            print(f"No improvement ({counter}/{patience})")

            if counter >= patience:

                print("\nEarly stopping triggered!")
                break

    print("\nTraining Complete")
    print("Best Validation Accuracy:", best_accuracy)

    model.load_state_dict(best_model_weights)

    return model

14. Stage 1 Training (Classifier Only)

In [ ]:

print("STAGE 1: TRAINING CLASSIFIER")

model = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=5
)

STAGE 1: TRAINING CLASSIFIER

Epoch 1/5
------------------------------
Train Loss: 1.3764
Train Acc : 0.5637
Val Loss : 0.7921
Val Acc  : 0.6819
Best model saved!

Epoch 2/5
------------------------------
Train Loss: 0.7200
Train Acc : 0.7070
Val Loss : 0.6716
Val Acc  : 0.7185
Best model saved!

Epoch 3/5
------------------------------
Train Loss: 0.6343
Train Acc : 0.7295
Val Loss : 0.6121
Val Acc  : 0.7435
Best model saved!

Epoch 4/5
------------------------------
Train Loss: 0.5890
Train Acc : 0.7362
Val Loss : 0.6009
Val Acc  : 0.7458
Best model saved!

Epoch 5/5
------------------------------
Train Loss: 0.5724
Train Acc : 0.7426
Val Loss : 0.5852
Val Acc  : 0.7509
Best model saved!

Training Complete
Best Validation Accuracy: tensor(0.7509, device='cuda:0', dtype=torch.float64)


### 15. Unfreeze Entire Network

In [17]:
for param in model.parameters():
    param.requires_grad = True

print("Entire network unfrozen!")

Entire network unfrozen!


### 16. Fine-Tuning Optimizer

In [18]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

### 17. Stage 2 Fine-Tuning

In [19]:
print("===================================")
print("STAGE 2: FINE TUNING")
print("===================================")

model = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=15
)

STAGE 2: FINE TUNING

Epoch 1/15
------------------------------
Train Loss: 0.3747
Train Acc : 0.8170
Val Loss : 0.2556
Val Acc  : 0.8760
Best model saved!

Epoch 2/15
------------------------------
Train Loss: 0.2129
Train Acc : 0.8796
Val Loss : 0.2007
Val Acc  : 0.8950
Best model saved!

Epoch 3/15
------------------------------
Train Loss: 0.1631
Train Acc : 0.9002
Val Loss : 0.1829
Val Acc  : 0.8982
Best model saved!

Epoch 4/15
------------------------------
Train Loss: 0.1391
Train Acc : 0.9119
Val Loss : 0.1791
Val Acc  : 0.9084
Best model saved!

Epoch 5/15
------------------------------
Train Loss: 0.1213
Train Acc : 0.9196
Val Loss : 0.1742
Val Acc  : 0.9081
No improvement (1/5)

Epoch 6/15
------------------------------
Train Loss: 0.1034
Train Acc : 0.9295
Val Loss : 0.1538
Val Acc  : 0.9196
Best model saved!

Epoch 7/15
------------------------------
Train Loss: 0.0900
Train Acc : 0.9372
Val Loss : 0.1961
Val Acc  : 0.9035
No improvement (1/5)

Epoch 8/15
----------------

### 18. Test Evaluation

In [ ]:
model.eval()

test_corrects = 0

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        test_corrects += torch.sum(preds == labels.data)

        all_predictions.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = test_corrects.double() / len(test_loader.dataset)

print("FINAL TEST RESULTS")

print(f"Test Accuracy: {test_accuracy:.4f}")

FINAL TEST RESULTS
Test Accuracy: 0.9266


### 19. Parameter Count

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("MODEL PARAMETERS")

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

MODEL PARAMETERS
Total Parameters     : 11,199,597
Trainable Parameters : 11,199,597


### 20. Save Final Model

In [22]:
torch.save(
    model.state_dict(),
    "resnetFineTunedFocalLoss.pth"
)

print("Final model saved!")

Final model saved!


### 22. Load Model Later (Optional)

In [ ]:
loaded_model = models.resnet18()

loaded_model.fc = nn.Linear(
    loaded_model.fc.in_features,
    45
)

loaded_model.load_state_dict(
    torch.load("final_pretrained_resnet18.pth")
)

loaded_model = loaded_model.to(device)

loaded_model.eval()

print("Model loaded successfully!")

C:\Users\Legion\AppData\Local\Temp\ipykernel_20732\1516620808.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("final_pretrained_resnet18.pth")


Model loaded successfully!
